# Shrink the Re-ID results so Claude can download them

Claude can only download Drive files under 10 MB; the `per_frame_tracks.csv` files in
`MyDrive/Playbook/reid_windows/` are 17-28 MB. This writes a compressed copy with only the columns needed
for identity scoring (`per_frame_tracks_small.csv.gz`, a few MB) next to each one. CPU runtime is enough,
about a minute. Running it again overwrites the copies. **Runtime -> Run all**; do not edit cells.

In [ ]:
# Cell 1 - compressed copies of the track files
import os, glob
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
DEST = '/content/drive/MyDrive/Playbook/reid_windows'
COLS = ['frame', 'raw_tracker_id', 'track_id', 'display_track_id', 'class_id', 'conf', 'x1', 'y1', 'x2', 'y2', 'x_m', 'y_m', 'team_id']
found = sorted(glob.glob(f'{DEST}/*/per_frame_tracks.csv'))
print('found:', [p.split('/')[-2] for p in found])
for p in found:
    t = pd.read_csv(p, low_memory=False)
    t = t[[c for c in COLS if c in t.columns]]
    t = t.round({'conf': 3, 'x1': 1, 'y1': 1, 'x2': 1, 'y2': 1, 'x_m': 0, 'y_m': 0})
    out = p.replace('per_frame_tracks.csv', 'per_frame_tracks_small.csv.gz')
    t.to_csv(out, index=False, compression='gzip')
    print(p.split('/')[-2], f'{os.path.getsize(out) / 1e6:.1f} MB')
print('Done. Tell Claude.')